In [ ]:
# ===============================================================
# 🧠 Week 2 — Data Preprocessing & Feature Engineering
# Project: HQNN for Catalyst Performance Prediction
# Author: Taofeek Sanyaolu
# ===============================================================

import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt
import mlflow

from fairchem.core.datasets.lmdb_dataset import LmdbDataset

# ---------------------------------------------------------------
# 1️⃣ Load LMDB Dataset
# ---------------------------------------------------------------
train_dir = r"E:\Computational Engineering\samples\is2res_lmdbs\is2res_train_val_test_lmdbs\data\is2re\100k\train"
config = {"src": train_dir, "train": True}
ds = LmdbDataset(config)
print(f"✅ Loaded {len(ds)} samples from {train_dir}")

print(f"✅ Dataset loaded: {len(ds)} samples")
print("Available keys:", list(ds[0].keys()))

# ---------------------------------------------------------------
# 2️⃣ Extract Features into DataFrame
# ---------------------------------------------------------------
records = []
for i in tqdm(range(min(len(ds), 10000)), desc="Extracting samples"):  # limit for manageable size
    s = ds[i]
    record = {
        "sid": s["sid"],
        "natoms": s["natoms"],
        "energy_init": float(s["y_init"]),
        "energy_relaxed": float(s["y_relaxed"]),
        "energy_diff": float(s["y_init"] - s["y_relaxed"]),
        "force_mean": np.linalg.norm(s["force"], axis=1).mean(),
        "force_std": np.linalg.norm(s["force"], axis=1).std(),
        "volume": abs(np.linalg.det(s["cell"])) if np.shape(s["cell"]) == (3,3) else np.nan,
        "atomic_number_mean": np.mean(s["atomic_numbers"]),
        "atomic_number_std": np.std(s["atomic_numbers"])
    }
    records.append(record)

df = pd.DataFrame(records)
print("\n✅ DataFrame created:")
display(df.head())

# ---------------------------------------------------------------
# 3️⃣ Handle Missing Values
# ---------------------------------------------------------------
print(f"Missing values before cleanup:\n{df.isnull().sum()}")
df = df.fillna(df.mean(numeric_only=True))
print(f"✅ Missing values handled. Shape: {df.shape}")

# ---------------------------------------------------------------
# 4️⃣ Normalize / Scale Features
# ---------------------------------------------------------------
features = ["natoms", "energy_init", "energy_relaxed", "energy_diff",
            "force_mean", "force_std", "volume", "atomic_number_mean", "atomic_number_std"]

scalers_dir = r"data/scalers"
os.makedirs(scalers_dir, exist_ok=True)

standard_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()

df_std = df.copy()
df_std[features] = standard_scaler.fit_transform(df[features])
df_mm = df.copy()
df_mm[features] = minmax_scaler.fit_transform(df[features])

import joblib
joblib.dump(standard_scaler, os.path.join(scalers_dir, "standard_scaler.pkl"))
joblib.dump(minmax_scaler, os.path.join(scalers_dir, "minmax_scaler.pkl"))

print("✅ Scaling complete and scalers saved!")

# ---------------------------------------------------------------
# 5️⃣ Split Dataset
# ---------------------------------------------------------------
train_df = df_std.sample(frac=0.8, random_state=42)
test_df = df_std.drop(train_df.index)
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

# ---------------------------------------------------------------
# 6️⃣ Save to Disk
# ---------------------------------------------------------------
out_dir = r"data/processed"
os.makedirs(out_dir, exist_ok=True)

train_df.to_csv(os.path.join(out_dir, "oc20_train.csv"), index=False)
test_df.to_csv(os.path.join(out_dir, "oc20_test.csv"), index=False)

train_df.to_parquet(os.path.join(out_dir, "oc20_train.parquet"), index=False)
test_df.to_parquet(os.path.join(out_dir, "oc20_test.parquet"), index=False)

print("✅ Saved processed datasets in both CSV and Parquet formats.")

# ---------------------------------------------------------------
# 7️⃣ Feature Correlation Analysis
# ---------------------------------------------------------------
plt.figure(figsize=(10,6))
sns.heatmap(df[features].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------
# 8️⃣ Optional — PCA Visualization
# ---------------------------------------------------------------
pca = PCA(n_components=2)
pca_features = pca.fit_transform(df_std[features])
plt.figure(figsize=(7,6))
sns.scatterplot(x=pca_features[:,0], y=pca_features[:,1], s=10, alpha=0.7)
plt.title("PCA Projection of Catalyst Features")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

# ---------------------------------------------------------------
# 9️⃣ Log preprocessing metadata in MLflow
# ---------------------------------------------------------------
mlflow.set_experiment("HQNN_Data_Preprocessing")
with mlflow.start_run(run_name="Week2_Preprocessing"):
    mlflow.log_param("num_samples", len(df))
    mlflow.log_param("num_features", len(features))
    mlflow.log_param("scaler_used", "StandardScaler & MinMaxScaler")
    mlflow.log_artifact(out_dir)
    mlflow.log_artifact(scalers_dir)
print("📊 MLflow run logged successfully.")
